## E-Commerce & Digital Sales Log Analysis

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
rows = 100000

data = {
    "Transaction_Date": pd.date_range(start="2026-01-01", periods=rows, freq="min").strftime("%Y-%m-%d %H:%M:%S"),
    "Product_Details": np.random.choice([" TECH-LAPTOP ", " tech-PHONE  ", " APPAREL-SHIRT ", " OFFICE-DESK "], size=rows),
    "Store_Location": np.random.choice([" MUM_IN ", " FRA_DE ", " YAK_RU "], size=rows),
    "Units_Sold": np.random.randint(1, 10, size=rows),
    "Unit_Price_USD": np.random.uniform(15.0, 1200.0, size=rows),
    "Unused_System_Code": ["SYS_X992_LOG_DUMP"] * rows
}

df_raw = pd.DataFrame(data)
df_raw.to_csv("raw_sales_data.csv", index=False)
print("Project Dataset 'raw_sales_data.csv' successfully created!")

Project Dataset 'raw_sales_data.csv' successfully created!


## Phase 1: Optimized Data Loading & Baseline Assessment

In [26]:
# Using the `usecols` parameter
keep_columns = ["Transaction_Date", "Product_Details", "Store_Location", "Units_Sold", "Unit_Price_USD"]
df_sales = pd.read_csv('raw_sales_data.csv', usecols = keep_columns)
print(df_sales)

          Transaction_Date  Product_Details Store_Location  Units_Sold  \
0      2026-01-01 00:00:00   APPAREL-SHIRT         MUM_IN            2   
1      2026-01-01 00:01:00     OFFICE-DESK         FRA_DE            8   
2      2026-01-01 00:02:00     TECH-LAPTOP         FRA_DE            7   
3      2026-01-01 00:03:00   APPAREL-SHIRT         FRA_DE            4   
4      2026-01-01 00:04:00   APPAREL-SHIRT         MUM_IN            4   
...                    ...              ...            ...         ...   
99995  2026-03-11 10:35:00   APPAREL-SHIRT         YAK_RU            5   
99996  2026-03-11 10:36:00     OFFICE-DESK         MUM_IN            9   
99997  2026-03-11 10:37:00   APPAREL-SHIRT         FRA_DE            5   
99998  2026-03-11 10:38:00     OFFICE-DESK         FRA_DE            7   
99999  2026-03-11 10:39:00     tech-PHONE          MUM_IN            2   

       Unit_Price_USD  
0         1080.229751  
1         1175.475117  
2          161.828273  
3          535.

In [3]:
print(df_sales.memory_usage(deep = True))

Index                   132
Transaction_Date    6800000
Product_Details     6249776
Store_Location      5700000
Units_Sold           800000
Unit_Price_USD       800000
dtype: int64


## Phase 2: Text Sanitization and Feature Extraction

In [4]:
# Clean up leading and trailing whitespaces from string columns and standardize them to lowercase
df_sales['Product_Details'] = df_sales['Product_Details'].str.strip().str.lower()
print(df_sales['Product_Details'].head())
df_sales['Store_Location'] = df_sales['Store_Location'].str.strip().str.lower()
print(df_sales['Store_Location'].head())

0    apparel-shirt
1      office-desk
2      tech-laptop
3    apparel-shirt
4    apparel-shirt
Name: Product_Details, dtype: object
0    mum_in
1    fra_de
2    fra_de
3    fra_de
4    mum_in
Name: Store_Location, dtype: object


In [5]:
# Split product details into independent 'Category' and 'Sub_Category' features
df_sales[['Category', 'Sub_Category']] = df_sales['Product_Details'].str.split('-', expand = True)
print(df_sales.head())

      Transaction_Date Product_Details Store_Location  Units_Sold  \
0  2026-01-01 00:00:00   apparel-shirt         mum_in           2   
1  2026-01-01 00:01:00     office-desk         fra_de           8   
2  2026-01-01 00:02:00     tech-laptop         fra_de           7   
3  2026-01-01 00:03:00   apparel-shirt         fra_de           4   
4  2026-01-01 00:04:00   apparel-shirt         mum_in           4   

   Unit_Price_USD Category Sub_Category  
0     1080.229751  apparel        shirt  
1     1175.475117   office         desk  
2      161.828273     tech       laptop  
3      535.189972  apparel        shirt  
4      978.678195  apparel        shirt  


## Phase 3: Advanced Memory Optimization & Downcasting

In [6]:
# Converting high-repetition string features into the categorical data type
df_sales['Store_Location'] = df_sales['Store_Location'].astype('category')
df_sales['Category'] = df_sales['Category'].astype('category')
df_sales['Sub_Category'] = df_sales['Sub_Category'].astype('category')

In [7]:
# Inspect numerical values to downcast default 64-bit data structures to lower bit-widths ('int8' and 'float32')
df_sales['Units_Sold'] = pd.to_numeric(df_sales['Units_Sold'] , downcast = 'integer')
df_sales['Unit_Price_USD'] = pd.to_numeric(df_sales['Unit_Price_USD'] , downcast = 'float')

In [8]:
print(df_sales.memory_usage(deep = True).sum())

13625750


## Phase 4: Time-Series Trend Analysis

In [9]:
# Convert the temporal index
df_sales['Transaction_Date'] = pd.to_datetime(df_sales['Transaction_Date'])
df_sales.set_index('Transaction_Date', inplace = True)
print(df_sales.head())

                    Product_Details Store_Location  Units_Sold  \
Transaction_Date                                                 
2026-01-01 00:00:00   apparel-shirt         mum_in           2   
2026-01-01 00:01:00     office-desk         fra_de           8   
2026-01-01 00:02:00     tech-laptop         fra_de           7   
2026-01-01 00:03:00   apparel-shirt         fra_de           4   
2026-01-01 00:04:00   apparel-shirt         mum_in           4   

                     Unit_Price_USD Category Sub_Category  
Transaction_Date                                           
2026-01-01 00:00:00     1080.229736  apparel        shirt  
2026-01-01 00:01:00     1175.475098   office         desk  
2026-01-01 00:02:00      161.828278     tech       laptop  
2026-01-01 00:03:00      535.190002  apparel        shirt  
2026-01-01 00:04:00      978.678223  apparel        shirt  


In [16]:
# Aggregate the financial performance
df_sales['Total_Revenue'] = df_sales['Units_Sold'] * df_sales['Unit_Price_USD']
print(df_sales['Total_Revenue'].head())

Transaction_Date
2026-01-01 00:00:00    2160.459473
2026-01-01 00:01:00    9403.800781
2026-01-01 00:02:00    1132.797974
2026-01-01 00:03:00    2140.760010
2026-01-01 00:04:00    3914.712891
Name: Total_Revenue, dtype: float32


In [22]:
daily_revenue = df_sales.resample('D')['Total_Revenue'].sum()
print(daily_revenue)

Transaction_Date
2026-01-01    4308071.50
2026-01-02    4503253.50
2026-01-03    4266961.50
2026-01-04    4497828.00
2026-01-05    4378555.00
                 ...    
2026-03-07    4409106.50
2026-03-08    4440913.00
2026-03-09    4254950.00
2026-03-10    4382035.00
2026-03-11    1975177.25
Freq: D, Name: Total_Revenue, Length: 70, dtype: float32


In [23]:
# Compute a rolling average to smooth out high-frequency volatility
daily_revenue_moving_avg = daily_revenue.rolling(window=3).mean()
print(daily_revenue_moving_avg)

Transaction_Date
2026-01-01             NaN
2026-01-02             NaN
2026-01-03    4.359429e+06
2026-01-04    4.422681e+06
2026-01-05    4.381115e+06
                  ...     
2026-03-07    4.353121e+06
2026-03-08    4.369641e+06
2026-03-09    4.368323e+06
2026-03-10    4.359299e+06
2026-03-11    3.537387e+06
Freq: D, Name: Total_Revenue, Length: 70, dtype: float64
